# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/sriteja/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/sriteja/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/sriteja/aibootcamp/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/sriteja/aibootcamp/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/sriteja/aibootcamp/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'a17d0d'. Skipping!
Property 'summary' already exists in node 'c6bb48'. Skipping!
Property 'summary' already exists in node '84c3df'. Skipping!
Property 'summary' already exists in node '61f737'. Skipping!
Property 'summary' already exists in node '926128'. Skipping!
Property 'summary' already exists in node 'e65df1'. Skipping!
Property 'summary' already exists in node '228f5d'. Skipping!
Property 'summary' already exists in node '07c8d3'. Skipping!
Property 'summary' already exists in node '3472a0'. Skipping!
Property 'summary' already exists in node '1b09bd'. Skipping!
Property 'summary' already exists in node '2dcf72'. Skipping!
Property 'summary' already exists in node '1cd85d'. Skipping!
Property 'summary' already exists in node '41a810'. Skipping!
Property 'summary' already exists in node 'b9e37c'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'c6bb48'. Skipping!
Property 'summary_embedding' already exists in node 'a17d0d'. Skipping!
Property 'summary_embedding' already exists in node '84c3df'. Skipping!
Property 'summary_embedding' already exists in node 'e65df1'. Skipping!
Property 'summary_embedding' already exists in node '1b09bd'. Skipping!
Property 'summary_embedding' already exists in node '2dcf72'. Skipping!
Property 'summary_embedding' already exists in node '41a810'. Skipping!
Property 'summary_embedding' already exists in node '3472a0'. Skipping!
Property 'summary_embedding' already exists in node '228f5d'. Skipping!
Property 'summary_embedding' already exists in node '07c8d3'. Skipping!
Property 'summary_embedding' already exists in node 'b9e37c'. Skipping!
Property 'summary_embedding' already exists in node '926128'. Skipping!
Property 'summary_embedding' already exists in node '1cd85d'. Skipping!
Property 'summary_embedding' already exists in node '61f737'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 478)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 478)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.


## Answer: 

1. SingleHopSpecificQuerySynthesizer: generates a specific question that can be answered by 1 document.

2. MultiHopAbstractQuerySynthesizer: generates a more abstract question that requires reasoning across multiple documents.

3. MultiHopSpecificQuerySynthesizer: generates a specific but multi-step question requiring multiple documents for the answer.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What are disbursements in the context of acade...,"[Chapter 1 Academic Years, Academic Calendars,...",Disbursements refer to the payments related to...,single_hop_specifc_query_synthesizer
1,What is 34 CFR 668.3(a)?,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(a) refers to the regulatory citat...,single_hop_specifc_query_synthesizer
2,Chapter 3 include clinical work?,[Inclusion of Clinical Work in a Standard Term...,Inclusion of Clinical Work in a Standard Term ...,single_hop_specifc_query_synthesizer
3,What are Non-Term Characteristics in relation ...,[Non-Term Characteristics A program that measu...,A program that measures progress in clock hour...,single_hop_specifc_query_synthesizer
4,What is a Direct Loen in the context of federa...,[both the credit or clock hours and the weeks ...,A Direct Loan is a type of federal student aid...,single_hop_specifc_query_synthesizer
5,disbursement timing in subscription programs a...,[<1-hop>\n\nboth the credit or clock hours and...,"In the first two subscription periods, student...",multi_hop_abstract_query_synthesizer
6,How do the disbursement rules and guidance for...,[<1-hop>\n\nboth the credit or clock hours and...,The disbursement rules and guidance for the TE...,multi_hop_abstract_query_synthesizer
7,How do clock-hour and non-term credit-hour pro...,[<1-hop>\n\nboth the credit or clock hours and...,In clock-hour and non-term credit-hour program...,multi_hop_abstract_query_synthesizer
8,Volume 2 Volume 7 what is that?,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The context mentions Volume 2 and Volume 7 as ...,multi_hop_specific_query_synthesizer
9,How do Chapters 2 and 3 collectively inform th...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Chapter 2 details the requirements for definin...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '43c4f4'. Skipping!
Property 'summary' already exists in node 'da1321'. Skipping!
Property 'summary' already exists in node '9afa74'. Skipping!
Property 'summary' already exists in node 'b95696'. Skipping!
Property 'summary' already exists in node '16a3e0'. Skipping!
Property 'summary' already exists in node 'd6a7a3'. Skipping!
Property 'summary' already exists in node '0e0979'. Skipping!
Property 'summary' already exists in node '3443bb'. Skipping!
Property 'summary' already exists in node 'c5f6dc'. Skipping!
Property 'summary' already exists in node 'a4f86b'. Skipping!
Property 'summary' already exists in node 'ee8ee6'. Skipping!
Property 'summary' already exists in node '4c0f0f'. Skipping!
Property 'summary' already exists in node '2788f8'. Skipping!
Property 'summary' already exists in node 'ccc267'. Skipping!


...

Property 'summary_embedding' already exists in node '43c4f4'. Skipping!
Property 'summary_embedding' already exists in node '9afa74'. Skipping!
Property 'summary_embedding' already exists in node '3443bb'. Skipping!
Property 'summary_embedding' already exists in node 'b95696'. Skipping!
Property 'summary_embedding' already exists in node 'c5f6dc'. Skipping!
Property 'summary_embedding' already exists in node 'd6a7a3'. Skipping!
Property 'summary_embedding' already exists in node 'da1321'. Skipping!
Property 'summary_embedding' already exists in node '4c0f0f'. Skipping!
Property 'summary_embedding' already exists in node '0e0979'. Skipping!
Property 'summary_embedding' already exists in node 'ee8ee6'. Skipping!
Property 'summary_embedding' already exists in node 'ccc267'. Skipping!
Property 'summary_embedding' already exists in node '16a3e0'. Skipping!
Property 'summary_embedding' already exists in node 'a4f86b'. Skipping!
Property 'summary_embedding' already exists in node '2788f8'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What is covered in Volume 2 regarding academic...,"[Chapter 1 Academic Years, Academic Calendars,...",Volume 2 includes information about academic y...,single_hop_specifc_query_synthesizer
1,What is 34 CFR 668.3(b)?,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(b) refers to the weeks of instruc...,single_hop_specifc_query_synthesizer
2,What information does Volume 8 provide regardi...,[Inclusion of Clinical Work in a Standard Term...,Volume 8 explains that clinical work conducted...,single_hop_specifc_query_synthesizer
3,Is the Federal Work-Study program considered a...,[Non-Term Characteristics A program that measu...,"No, the Federal Work-Study (FWS) program is an...",single_hop_specifc_query_synthesizer
4,How does including clinical work in standard t...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Including clinical work in standard term perio...,multi_hop_abstract_query_synthesizer
5,How does the accumulation of cummulative credi...,[<1-hop>\n\nboth the credit or clock hours and...,The disbursement of FSEOG funds depends on mee...,multi_hop_abstract_query_synthesizer
6,how is clinical work inclusion in term periods...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work in standard ter...,multi_hop_abstract_query_synthesizer
7,How does measuring academic progress in clock ...,[<1-hop>\n\nInclusion of Clinical Work in a St...,"According to the context, programs that measur...",multi_hop_abstract_query_synthesizer
8,How does Volume 8 relate to the inclusion of c...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Volume 8 provides guidance on including clinic...,multi_hop_specific_query_synthesizer
9,How do Chapters 2 and 3 relate to the academic...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Chapter 2 discusses the minimum instructional ...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [16]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [17]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [18]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [19]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [20]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [21]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [23]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [25]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [26]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available mentioned in the context are:\n\n- Direct Subsidized Loan  \n- Direct Unsubsidized Loan  \n- Direct PLUS Loan (student Federal PLUS Loan or parent PLUS Loan)  \n- Subsidized and Unsubsidized Federal Stafford Loans (made under the discontinued Federal Family Education Loan (FFEL) Program before July 1, 2010)  \n- Federal SLS Loans (also under FFEL Program)  \n- Federal PLUS Loans (under FFEL Program before July 1, 2010)  \n- Direct Consolidation Loans  \n- Federal Consolidation Loans (under FFEL Program)  \n\nAdditionally, Direct Subsidized Loans are only available to undergraduate students, while graduate or professional students are eligible for Direct Unsubsidized Loans and Direct PLUS Loans.\n\nPreparatory coursework and teacher certification coursework loans are also mentioned, indicating eligibility for certain loans in preparatory courses documented as necessary for enrollment.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [27]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [28]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:

## Answer: 

qa: how accurate/relevant the output is.

labeled helpfulness: based on the reference answer, how helpful the output is to the user.

empathy: essentially how kind, warm, including, and respectful the output is.

## LangSmith Evaluation

In [29]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'proper-stew-76' at:
https://smith.langchain.com/o/ba8cbc21-d8e0-4bdb-9236-2ad4af22bdb3/datasets/f2b57cf7-7f36-4450-80d7-2b6a72c353e0/compare?selectedSessions=68928e1d-4a24-4992-b095-0195445edfaf




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Volume 2 and Volume 8 relate to the inc...,I don't know.,None,Volume 2 discusses the requirements for defini...,0,0,0,1.547845,1daee8cc-434c-4a85-ba5f-dd85023f005a,16f8ed53-07b6-4fb1-ba09-10ef56536ad3
1,"In chapter 3, how does inclusion of clinical w...","Based on the provided context, clinical work t...",None,Chapter 3 discusses that clinical work include...,1,0,0,5.751240,ddcf6215-7bf0-4842-9ff6-9a0cfbc95d73,63926c95-44aa-4277-9392-fedce6e50957
2,How do Chapters 2 and 3 relate to the academic...,I don't know.,None,Chapter 2 discusses the minimum instructional ...,0,0,0,2.172165,4acc7eee-69f3-4172-aacc-9e2d7c9d0103,6a8ec3ec-22e4-4335-b8e1-2c21d599e92e
3,How does Volume 8 relate to the inclusion of c...,"Based on the provided context, Volume 8 discus...",None,Volume 8 provides guidance on including clinic...,1,0,0,5.342857,7d3babfc-5a13-434b-ab91-100e31a96d99,db76fc70-3b10-42ce-888c-8cafa140b784
4,How does measuring academic progress in clock ...,Measuring academic progress in clock hours res...,None,"According to the context, programs that measur...",1,1,0,4.740739,89295d25-021c-45d9-b77e-8282d8582e83,004295c3-6419-4528-adef-75f23efc899b
5,how is clinical work inclusion in term periods...,"Based on the provided context, clinical work i...",None,The inclusion of clinical work in standard ter...,0,0,0,2.361274,37f19b70-4f1b-4153-b98a-012eeba95564,e8e40759-af62-47ea-9fce-3a0718c02ae8
6,How does the accumulation of cummulative credi...,"Based on the provided context, the accumulatio...",None,The disbursement of FSEOG funds depends on mee...,1,1,0,13.120952,ccbc9b14-afbc-49ac-84a2-d36515f01abe,ec93032a-7b85-4505-88fe-1b5324295398
7,How does including clinical work in standard t...,Including clinical work in standard term perio...,None,Including clinical work in standard term perio...,1,0,0,6.966071,4c081b16-41e2-4083-8b4f-f869c01a59f0,7341f0b8-d8ee-4256-91aa-ace4f31abc8e
8,Is the Federal Work-Study program considered a...,"No, the Federal Work-Study (FWS) Program is no...",None,"No, the Federal Work-Study (FWS) program is an...",1,1,0,1.662841,d39753f4-6df4-4167-ac93-b86a7c95f023,08484dd3-1da6-4167-a14a-2dd576ac7c3b
9,What information does Volume 8 provide regardi...,Volume 8 provides guidance on the inclusion of...,None,Volume 8 explains that clinical work conducted...,1,1,0,6.094359,f04dc3c2-5018-4364-a272-1aee747ff1f5,ea43f50e-25b9-4aaf-b10e-7c3b9676669c


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [30]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [31]:
rag_documents = docs

In [32]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

## Answer: 

Modifying our chunk size modifies the performance of our application because essentially, changing chunk size changes how much context the model sees at once. If the chunks are too small, important context might be split up, and if they are too large, the retriever might pull in too much irrelevant information. That is why the right chunk size depends on the specific use-case or data, and finding the right size optimizes for model output clarity.

In [33]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

## Answer: 

Modifying our embedding model modifies the performance of our application because the embedding model is related to how well the AI system matches the question to relevant and semantically similar vectors (text chunks), so a better embedding model can capture more aspects (dimensions) of the data and hence find better and more relevant information.

In [34]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [35]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [36]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [37]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question. Based on the context you provided, there are several kinds of loans available to students:\n\n1. **Direct Subsidized Loans** – These loans are based on the student’s financial need and have a maximum eligibility amount tied to that need.\n\n2. **Direct Unsubsidized Loans** – These loans are available regardless of financial need and can be used to complement subsidized loans or cover unmet need.\n\n3. **Direct PLUS Loans** – These are loans that parents of dependent students can take out to help pay for the student’s cost of attendance, assuming eligibility. There is no fixed loan limit for PLUS Loans, but the amount cannot exceed the student’s cost of attendance minus other financial aid.\n\nAdditionally, if a dependent student’s parent cannot obtain a Direct PLUS Loan, the student may be eligible for additional Direct Unsubsidized Loan funds.\n\nI hope this helps clarify the types of loans available. If you have more questions or need further explanation

Finally, we can evaluate the new chain on the same test set!

In [38]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'unique-speed-35' at:
https://smith.langchain.com/o/ba8cbc21-d8e0-4bdb-9236-2ad4af22bdb3/datasets/f2b57cf7-7f36-4450-80d7-2b6a72c353e0/compare?selectedSessions=e4066f4f-f7cc-4326-9d5a-125bf4d56968




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Volume 2 and Volume 8 relate to the inc...,Thank you for your thoughtful question. From t...,None,Volume 2 discusses the requirements for defini...,0,0,1,3.679363,1daee8cc-434c-4a85-ba5f-dd85023f005a,7d5af36f-9ec7-4508-b8e4-82ad0f2503fa
1,"In chapter 3, how does inclusion of clinical w...",Thank you for your thoughtful question. From t...,None,Chapter 3 discusses that clinical work include...,1,0,1,5.960066,ddcf6215-7bf0-4842-9ff6-9a0cfbc95d73,1c2e24b9-0d14-4a3e-99b6-f70b59649d81
2,How do Chapters 2 and 3 relate to the academic...,Thank you for your thoughtful question. It’s c...,None,Chapter 2 discusses the minimum instructional ...,0,0,1,8.028343,4acc7eee-69f3-4172-aacc-9e2d7c9d0103,18cb0716-daeb-4d45-b134-4a587edf44aa
3,How does Volume 8 relate to the inclusion of c...,Thank you for your thoughtful question. Based ...,None,Volume 8 provides guidance on including clinic...,1,0,1,4.917426,7d3babfc-5a13-434b-ab91-100e31a96d99,0a3f798a-43d6-4f19-8ef1-0374903c6f03
4,How does measuring academic progress in clock ...,Thank you for your thoughtful question. Based ...,None,"According to the context, programs that measur...",1,1,1,11.709416,89295d25-021c-45d9-b77e-8282d8582e83,f801cd4a-c1b8-4650-9485-49d0a3e87fb2
5,how is clinical work inclusion in term periods...,Thank you for your thoughtful question about h...,None,The inclusion of clinical work in standard ter...,1,1,1,3.892517,37f19b70-4f1b-4153-b98a-012eeba95564,40b1a145-5cf4-4a7f-a2f4-bb7323583917
6,How does the accumulation of cummulative credi...,Thank you for your thoughtful question. From t...,None,The disbursement of FSEOG funds depends on mee...,1,0,1,8.366384,ccbc9b14-afbc-49ac-84a2-d36515f01abe,f1c56c00-f23c-444d-b032-97c204893187
7,How does including clinical work in standard t...,Thank you for your thoughtful question. From t...,None,Including clinical work in standard term perio...,1,0,1,18.423703,4c081b16-41e2-4083-8b4f-f869c01a59f0,e708e2c2-62d6-401c-b5c2-bb909add0556
8,Is the Federal Work-Study program considered a...,Thank you for your question. Based on the cont...,None,"No, the Federal Work-Study (FWS) program is an...",1,1,1,2.599279,d39753f4-6df4-4167-ac93-b86a7c95f023,38029da6-4c4d-40a6-9b3c-ae6ad09f387a
9,What information does Volume 8 provide regardi...,Thank you for your thoughtful question. Based ...,None,Volume 8 explains that clinical work conducted...,1,0,1,4.062317,f04dc3c2-5018-4364-a272-1aee747ff1f5,634998ea-9f24-4614-90b5-0e06aab6022d


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

## Answer:

Both screenshots are provided in the 07_Synthetic_Data... folder. 

What changed:

1. Evidently, the aggregate empathy score went from 0 to 1, because we have changed our system prompt to optimize for providing empathetic responses.

2. The helpfulness score went from an average of ~0.42 to 0.25, maybe because of the fact that we have specifically given instructions to the LLM to dial in on empathy, which could have taken up most of the LLM's focus, letting it perform worse in the helpfulness metric.

3. The latency went up, and this is because of added instructions to make the messages friendly and kind.